# Listing t-value of t-tests
## TWQ Only

In [1]:
from scipy.stats import shapiro, mannwhitneyu
%cd "/home/adam/personal/msc/thesis/Data and code"
import pickle
from adamthesis.gridsearch import *

suffix = "10folding_with_stats"
results = {"individual-491": {}, "group-491": {}, "individual-492": {}, "group-492": {}}

for target in ["individual-491", "group-491", "individual-492", "group-492"]:
    for experiment in ["twq","academic","both","prereqs"]:
        with open(f"logs/pickles/{target}_{experiment}_{suffix}.pkl", "rb") as f:
            results[target][experiment] = pickle.load(f)

# save results to a new pickle for easier access in the future
with open(f"logs/pickles/summary_results_{suffix}.pkl", "wb") as f:
    pickle.dump(results, f)

def test_normality(fold_results1, fold_results2):
    """A function which tests normality via the Shapiro-Wilk test to consider t-test validity
    Generally, we are testing the results of 1 set of folds against a baseline on the same data, hence we're using the paired t-tests.
    """
    diff = [a - b for a, b in zip(fold_results1, fold_results2)]
    stat, p = shapiro(diff)
    if p > 0.05:
        return True
    else:
        return False

def print_stats(model):
    print("| Target | Model | Val Acc | Is Normal? | t-value | p-value |")
    print("| --- | --- | --- | --- | --- | --- |")
    for target in ["individual-491", "group-491", "individual-492", "group-492"]:
        # find best performing model
        max_score = max(r["validation_score"] for r in results[target][model].validation_results)
        both_comparison_string = [r["model"] for r in results[target][model].validation_results if r["validation_score"] == max_score][0]
        stats_dict = results[target][model].statistical_results[both_comparison_string]
        best_model_val_folds = [r["fold_val_scores"] for r in results[target][model].results if r["model"] == both_comparison_string][0]
        baseline_model_val_folds = [r["fold_val_scores"] for r in results[target][model].results if r["model"] == "baseline_classifier"][0]
        is_normal = test_normality(best_model_val_folds, baseline_model_val_folds)
        print(f"| {target} | {both_comparison_string} | {max_score:.4f} | {is_normal} | {stats_dict['t_statistic']:.4f} | {stats_dict['t_pvalue']:.4f} |")


/home/adam/personal/msc/thesis/Data and code


In [4]:
print("== TWQ Only ==")
print_stats("twq")

== TWQ Only ==
| Target | Model | Val Acc | Is Normal? | t-value | p-value |
| --- | --- | --- | --- | --- | --- |
| individual-491 | svc | 0.6800 | True | 2.1669 | 0.0292 |
| group-491 | KNN | 0.7167 | True | 2.0250 | 0.0368 |
| individual-492 | svc | 0.6700 | True | 5.0747 | 0.0003 |
| group-492 | logistic_regression | 0.7167 | True | 2.0555 | 0.0350 |


## Academic Only

In [5]:
print("== Academic Only ==")
print_stats("academic")

== Academic Only ==
| Target | Model | Val Acc | Is Normal? | t-value | p-value |
| --- | --- | --- | --- | --- | --- |
| individual-491 | random_forest | 0.7400 | False | 6.1942 | 0.0001 |
| group-491 | logistic_regression | 0.8500 | True | 3.8988 | 0.0018 |
| individual-492 | logistic_regression | 0.6900 | True | 4.1461 | 0.0012 |
| group-492 | random_forest | 0.8333 | True | 3.1225 | 0.0061 |


## Academic + TWQ

In [6]:
print("== Academic + TWQ ==")
print_stats("both")

== Academic + TWQ ==
| Target | Model | Val Acc | Is Normal? | t-value | p-value |
| --- | --- | --- | --- | --- | --- |
| individual-491 | KNN | 0.7800 | True | 5.2842 | 0.0003 |
| group-491 | logistic_regression | 0.8500 | True | 3.8988 | 0.0018 |
| individual-492 | KNN | 0.7800 | True | 6.3317 | 0.0001 |
| group-492 | svc | 0.7833 | True | 2.1809 | 0.0285 |


## Prereqs Only


In [7]:
print("== Prereqs Only ==")
print_stats("prereqs")

== Prereqs Only ==
| Target | Model | Val Acc | Is Normal? | t-value | p-value |
| --- | --- | --- | --- | --- | --- |
| individual-491 | DecisionTree | 0.6900 | False | 3.0736 | 0.0066 |
| group-491 | KNN | 0.7500 | True | 2.5501 | 0.0156 |
| individual-492 | svc | 0.6700 | True | 2.6842 | 0.0125 |
| group-492 | random_forest | 0.6500 | True | 1.1677 | 0.1365 |


# T-test Matrix

In [8]:
from scipy.stats import ttest_rel

for target in ["individual-491", "group-491", "individual-492", "group-492"]:
    print(f"== {target}: Comparison Matrix==")
        #Comparing all 10 folds in a matrix, best model in academic, best model in twq, best model in both
    best_twq_model = [r["model"] for r in results[target]["twq"].validation_results if r["validation_score"] == max(r["validation_score"] for r in results[target]["twq"].validation_results)][0]
    best_academic_model = [r["model"] for r in results[target]["academic"].validation_results if r["validation_score"] == max(r["validation_score"] for r in results[target]["academic"].validation_results)][0]
    best_both_model = [r["model"] for r in results[target]["both"].validation_results if r["validation_score"] == max(r["validation_score"] for r in results[target]["both"].validation_results)][0]
    best_twq_folds = [r["fold_val_scores"] for r in results[target]["twq"].results if r["model"] == best_twq_model][0]
    best_academic_folds = [r["fold_val_scores"] for r in results[target]["academic"].results if r["model"] == best_academic_model][0]
    best_both_folds = [r["fold_val_scores"] for r in results[target]["both"].results if r["model"] == best_both_model][0]
    print("\n| Fold | best TWQ model acc.| best Academic model acc. | best Academic + TWQ model acc. |")
    print("| --- | --- | --- | --- |")
    for i in range(10):
        max_academic_twq_score = max(best_academic_folds[i],best_twq_folds[i])
        if best_both_folds[i] > max_academic_twq_score:
            both_comparison_string = "*(Best)*"
        elif best_academic_folds[i] == max_academic_twq_score:
            both_comparison_string = "(Tied)"
        else:
            both_comparison_string = ""
        print(f"| Fold {i+1} | {best_twq_folds[i]:.2f} | {best_academic_folds[i]:.2f} | {best_both_folds[i]:.2f} {both_comparison_string} |")
    print()
    print("T-test results for all folds on academic vs both, twq vs both, and max(academic,twq) vs both")
    print("| Comparison | Is Normal? | T-test t-value | T-test p-value |")
    print("| --- | --- | --- | --- |")
    for comparison in [(best_academic_folds, best_both_folds, "Academic vs TWQ+Academic"), (best_twq_folds, best_both_folds, "TWQ vs TWQ+Academic"), ( [max(a,b) for a,b in zip(best_academic_folds, best_twq_folds)], best_both_folds, "Max(Academic,TWQ) vs Academic+TWQ")]:
        is_normal = test_normality(comparison[0], comparison[1])
        stat, p = ttest_rel(comparison[1], comparison[0], alternative="greater")
        if (is_normal):
            print(f"| {comparison[2]} | {is_normal} | {stat:.4f} | {p:.4f} |")
        else:
            stat, p = mannwhitneyu(comparison[1], comparison[0], alternative="greater")
            print(f"| {comparison[2]} | {is_normal} | {stat:.4f} | {p:.4f} |")


    print()

== individual-491: Comparison Matrix==

| Fold | best TWQ model acc.| best Academic model acc. | best Academic + TWQ model acc. |
| --- | --- | --- | --- |
| Fold 1 | 0.60 | 0.80 | 0.70 (Tied) |
| Fold 2 | 0.80 | 0.60 | 0.70  |
| Fold 3 | 0.60 | 0.80 | 0.90 *(Best)* |
| Fold 4 | 0.70 | 0.90 | 0.90 (Tied) |
| Fold 5 | 0.50 | 0.70 | 0.70 (Tied) |
| Fold 6 | 0.80 | 0.70 | 0.70  |
| Fold 7 | 0.70 | 0.70 | 0.80 *(Best)* |
| Fold 8 | 0.70 | 0.70 | 0.70 (Tied) |
| Fold 9 | 0.60 | 0.70 | 0.70 (Tied) |
| Fold 10 | 0.80 | 0.80 | 1.00 *(Best)* |

T-test results for all folds on academic vs both, twq vs both, and max(academic,twq) vs both
| Comparison | Is Normal? | T-test t-value | T-test p-value |
| --- | --- | --- | --- |
| Academic vs TWQ+Academic | True | 1.5000 | 0.0839 |
| TWQ vs TWQ+Academic | True | 2.3717 | 0.0209 |
| Max(Academic,TWQ) vs Academic+TWQ | True | 0.3180 | 0.3789 |

== group-491: Comparison Matrix==

| Fold | best TWQ model acc.| best Academic model acc. | best Academic + TW